# Face model v2 — round 3: EAC (Erasing Attention Consistency, Zhang et al. ECCV 2022)
Train-time only: random erasing + the class-attention map of an image must equal the mirrored map of its mirror image. The exported model is unchanged (same latency).

Reference points on the same validation split: v1 recipe on Colab = 0.647 (3 seeds, sd 0.002); res112 = 0.6634; mix_e30 = 0.6606.

Needs in `MyDrive/face-v2/`: `Face-Lab-code-r3.zip`, the FER2013 zip, `out/face_v2_results_r2.zip`. Runtime → T4 GPU.

In [ ]:
import shutil, subprocess
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi','-L'],capture_output=True,text=True).stdout)
else:
    raise SystemExit('NO GPU: Runtime > Change runtime type > T4 GPU, then Run all again')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import glob, os, zipfile
D='/content/drive/MyDrive/face-v2'
os.makedirs('/content/Main_demo', exist_ok=True)
zipfile.ZipFile(f'{D}/Face-Lab-code-r3.zip').extractall('/content/Main_demo')
fer = [p for p in glob.glob(f'{D}/*.zip') if 'Face-Lab' not in p][0]
zipfile.ZipFile(fer).extractall('/content/Main_demo/FER2013-Data')
zipfile.ZipFile(f'{D}/out/face_v2_results_r2.zip').extractall('/content/Main_demo/Face-Lab')   # rounds 1-2 results/ for the table
for s in ('train','test'):
    print(s, sum(len(f) for _,_,f in os.walk(f'/content/Main_demo/FER2013-Data/{s}')))   # expect 28709 / 7178
%cd /content/Main_demo/Face-Lab
print(sorted(os.listdir('results')))

In [ ]:
BASE = '--arch efficientnet_b0 --epochs 20 --head-epochs 1 --unfreeze 0.0 --split-seed 42 --seed 42'
EXPS = {
  'v2_eac1':        '--res 96 --eac 1',
  'v2_eac5':        '--res 96 --eac 5',     # the paper's weight
  'v2_eac5_res112': '--res 112 --eac 5',
}
for tag, extra in EXPS.items():
    cmd = f'python train.py {BASE} {extra} --tag {tag}'
    print('>>', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    lines = r.stdout.strip().splitlines()
    print(chr(10).join(lines[-3:]) if lines else '', r.stderr[-800:] if r.returncode else '', flush=True)


In [ ]:
import json, glob, statistics as st
rows = {}
for f in sorted(glob.glob('results/v2_*.json')):
    r = json.load(open(f)); v = r['val']; pc = v['per_class']
    rows[r['tag']] = (r['best_val_macro_f1'], v['accuracy'], pc['sad']['f1'], pc['fear']['f1'], pc['neutral']['f1'], r['best_epoch'], r['train_minutes'])
seeds = [rows[t][0] for t in ('v2_seed42', 'v2_seed1', 'v2_seed2') if t in rows]
base = st.mean(seeds) if seeds else 0.0
print('baseline = v1 recipe on Colab, mean of', len(seeds), 'seeds:', round(base, 4))
print('run'.ljust(16), 'macroF1  vsBase   acc    sad   fear  neutr  ep   min')
for t, (mf, a, sa, fe, ne, ep, mn) in sorted(rows.items(), key=lambda kv: -kv[1][0]):
    print(t.ljust(16), '%.4f  %+.4f  %.3f  %.3f  %.3f  %.3f  %3d  %4.1f' % (mf, mf - base, a, sa, fe, ne, ep, mn))

In [ ]:
OUT='/content/drive/MyDrive/face-v2/out'
shutil.make_archive('/content/face_v2_results_r3','zip','/content/Main_demo/Face-Lab','results')
shutil.copy('/content/face_v2_results_r3.zip', OUT)
for t in EXPS:
    if os.path.exists(f'runs/{t}/best.pt'):
        os.makedirs(f'{OUT}/{t}', exist_ok=True)
        b = open(f'runs/{t}/best.pt','rb').read()
        for i in range(0, len(b), 9000000):                     # parts < 10 MB so they can be pulled back to the laptop
            open(f'{OUT}/{t}/best.pt.part{i // 9000000}','wb').write(b[i:i + 9000000])
print(sorted(os.listdir(OUT)))